In [84]:
import json
from collections import defaultdict

# JSON 파일 로드
file_path = '/content/1. 서울_열린데이터/02. 일반행정/TEXT_NL2SQL_label_seouldata_publicadministration.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# db_id 별로 데이터 개수 집계
db_id_count = defaultdict(int)
for item in data['data']:
    db_id_count[item['db_id']] += 1

# 결과 출력
print("db_id 별 데이터 개수:")
for db_id, count in db_id_count.items():
    print(f"{db_id}: {count}개")

# # 선택적으로 결과를 파일에 저장
# output_path = '/content/db_id_counts.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(db_id_count, f, ensure_ascii=False, indent=4)

# print(f"db_id 별 데이터 개수가 {output_path}에 저장되었습니다.")


db_id 별 데이터 개수:
seouldata_publicadministration_978: 18개
seouldata_publicadministration_1283: 17개
seouldata_publicadministration_314: 17개
seouldata_publicadministration_20: 13개
seouldata_publicadministration_1292: 18개
seouldata_publicadministration_5369: 20개
seouldata_publicadministration_1272: 19개
seouldata_publicadministration_1290: 13개
seouldata_publicadministration_1265: 16개
seouldata_publicadministration_1303: 19개
seouldata_publicadministration_1256: 18개
seouldata_publicadministration_1286: 17개
seouldata_publicadministration_359: 10개
seouldata_publicadministration_1398: 15개
seouldata_publicadministration_5313: 13개
seouldata_publicadministration_1227: 15개
seouldata_publicadministration_1217: 14개
seouldata_publicadministration_91: 11개
seouldata_publicadministration_5388: 18개
seouldata_publicadministration_1600: 19개
seouldata_publicadministration_1379: 3개
seouldata_publicadministration_301: 1개


# 서울_열린데이터 - 일반행정

In [122]:
import pandas as pd

index_names = ["accuracy", "time"]
column_names = ["gpt-3.5-turbo", "gpt-4-turbo", "gpt-4o-mini"]

df = pd.DataFrame(index=index_names, columns=column_names)

In [126]:
df['gpt-3.5-turbo'] = ['50.31%', '0.68s']

In [128]:
df['gpt-4-turbo'] = ['62.04%', '1.78s']

In [129]:
df['gpt-4o-mini'] = ['55.86%', '1.00s']

In [130]:
df

,gpt-3.5-turbo,gpt-4-turbo,gpt-4o-mini
accuracy,50.31%,62.04%,55.86%
time,0.68s,1.78s,1.00s


In [5]:
pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.54.5
    Uninstalling openai-1.54.5:
      Successfully uninstalled openai-1.54.5


In [98]:
import json

labeling_data_path = '/content/1. 서울_열린데이터/02. 일반행정/TEXT_NL2SQL_label_seouldata_publicadministration.json'

with open(labeling_data_path, 'r', encoding='utf-8') as f:
    labeling_data = json.load(f)

# 데이터 개수 확인
data_count = len(labeling_data['data'])
print(f"데이터 개수: {data_count}")


데이터 개수: 324


In [ ]:
import json
import openai
import time

openai.api_key = "OPEN_API_KEY"

labeling_data_path = '/content/1. 서울_열린데이터/02. 일반행정/TEXT_NL2SQL_label_seouldata_publicadministration.json'
db_info_path = '/content/1. 서울_열린데이터/02. 일반행정/seouldata_publicadministration_db_annotation.json'

with open(labeling_data_path, 'r', encoding='utf-8') as f:
    labeling_data = json.load(f)

with open(db_info_path, 'r', encoding='utf-8') as f:
    db_info = json.load(f)

def get_db_info(db_id):   # db_id에 해당하는 데이터베이스 정보를 반환
    for db in db_info['data']:
        if db['db_id'] == db_id:
            return db
    return None

def generate_sql_prompt(utterance, db_id, cols):
    db_metadata = get_db_info(db_id)
    if not db_metadata:
        print(f"{db_id}에 대한 DB 정보 없음.")
        return "", 0

    table_names = db_metadata['table_names_original']
    column_names = db_metadata['column_names_original']
    column_info = db_metadata['column_names']

    # 컬럼 매핑 (영어 컬럼명만 사용)
    column_mapping = "\n".join([f"- {col[1]}" for col in column_names if col[0] != -1])

    selected_columns = ", ".join([
        column_names[col['column_index']][1] for col in cols
    ])

    # SQL 프롬프트
    prompt = f"""
    너는 SQL 전문가 AI야. 사용자의 요청에 맞게 정확한 SQL 쿼리를 작성해.

    **데이터베이스 정보**:
    - 테이블 이름: {', '.join(table_names)}
    - 컬럼 정보 (영어):
    {column_mapping}

    - 컬럼에 대한 부가적 정보 : {column_info}

    **사용자 요청**:
    "{utterance}"

    **규칙**:
    1. 테이블 이름과 영어 컬럼명만 사용해 SQL을 작성해. 번역된 한국어 컬럼명은 절대 사용하지 마.
    2. SELECT 절에는 데이터베이스에 있는 정확한 column name만을 사용해.
    3. WHERE, GROUP BY, HAVING 절은 요청의 조건을 바탕으로 필요할 경우에만 작성하고, 불필요한 조건은 포함하지 마.
    4. WHERE 조건에서는 `=`를 기본적으로 사용하고, 문자열 포함 검색일 경우에만 `LIKE`를 사용해.
      - 문자열 비교 시 공백 처리와 대소문자 차이에 주의해.
    5. COUNT와 GROUP BY 같은 집계 함수는 항상 정확히 매핑된 컬럼으로 작성해.
    6. 요청에 맞지 않는 추가적인 조건이나 불필요한 컬럼은 절대 포함하지 마.
    7. SQL 코드만 반환해. 추가 설명이나 불필요한 내용은 포함하지 마.
    8. 그리고 내가 컬럼에 대한 부가적 정보로 준게 영어 columns에 대한 한국어 설명이라고 생각하면 돼. 잘 매칭해서 정확한 SQL문을 생성해

    **예시**:
    사용자 요청: "3년제 전문대학의 학교 이름을 알려줘."
    SQL: SELECT NAME_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE CATE1_NAME LIKE '전문대학%' AND CATE1_NAME LIKE '%3년제%'

    사용자 요청: "중곡동 31-7에 위치한 와이파이 이름을 보여줘."
    SQL: SELECT X_SWIFI_MAIN_NM FROM TB_PUBLIC_WIFI_INFO_G_J WHERE X_SWIFI_ADRES1 = '중곡동31-7'

    이제 SQL 쿼리를 작성해:
    "{utterance}"
    """
    return prompt

def generate_sql(utterance, db_id, cols):

    prompt = generate_sql_prompt(utterance, db_id, cols)
    if not prompt:
        return "", 0

    try:
        start_time = time.time()
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=1
        )
        end_time = time.time()
        response_time = end_time - start_time
        generated_sql = response['choices'][0]['message']['content'].strip()
        return generated_sql, response_time
    except Exception as e:
        print(f"API 호출 에러: {e}")
        return "", 0




In [120]:
def evaluate_semantic_accuracy_and_time(labeling_data):   # 정확도, 평균 시간 평가

    total_count = len(labeling_data['data'])
    correct_count = 0
    total_time = 0

    for idx, item in enumerate(labeling_data['data'], start=1):
        utterance = item['utterance']
        db_id = item['db_id']
        cols = item['cols']
        true_query = item['query']

        # SQL 쿼리 생성
        generated_query, elapsed_time = generate_sql(utterance, db_id, cols)
        total_time += elapsed_time

        # 정확도 평가 프롬프트
        prompt = f"""
        너는 SQL 전문가 AI야. 두 개의 SQL 쿼리가 같은 의미를 가지는지 평가해줘.
        **SQL 쿼리 1**:
        {true_query}

        **SQL 쿼리 2**:
        {generated_query}

        **지시사항**:
        1. 두 쿼리가 같은 결과를 반환하면 "True"라고 응답해.
        2. 다른 결과를 반환하면 "False"라고 응답해.
        3. 추가 설명은 절대 하지 마. 반드시 True 또는 False만 반환해.
        """

        try:
            response = openai.ChatCompletion.create(
                model="gpt-4-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            evaluation_result = response['choices'][0]['message']['content'].strip()

            if evaluation_result == "True":
                correct_count += 1

            if idx % 10 == 0:  # 10번마다 진행 상황 출력
                print(f"진행 상황: {idx}/{total_count}개 완료")

            print(f"문장: {utterance}")
            print(f"생성된 쿼리: {generated_query}")
            print(f"정답 쿼리: {true_query}")
            print(f"평가 결과: {evaluation_result}")
            print("-" * 50)

        except Exception as e:
            print(f"4-turbo 호출 에러: {e}")

    accuracy = correct_count / total_count * 100
    average_time = total_time / total_count

    print(f"정확도: {accuracy:.2f}%")
    print(f"평균 응답 시간: {average_time:.2f}초")

    return accuracy, average_time

accuracy, avg_time = evaluate_semantic_accuracy_and_time(labeling_data)


문장: 구로구에 위치한 대학교를 알려줘
생성된 쿼리: SQL: SELECT NAME_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE H_KOR_GU = '구로구' AND CATE1_NAME = '대학'
정답 쿼리: SELECT NAME_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE H_KOR_GU = '구로구'
평가 결과: False
--------------------------------------------------
문장: 성균관대학교의 주소를 알려줘
생성된 쿼리: SQL: SELECT ADD_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE NAME_KOR = '성균관대학교'
정답 쿼리: SELECT ADD_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE NAME_KOR = '성균관대학교'
평가 결과: True
--------------------------------------------------
문장: 학교 이름이 한국으로 시작하는 곳의 주소를 알려줘
생성된 쿼리: SQL: SELECT ADD_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE NAME_KOR LIKE '한국%'
정답 쿼리: SELECT ADD_KOR FROM SEBC_COLLEGE_INFO_KOR WHERE NAME_KOR LIKE '한국%'
평가 결과: True
--------------------------------------------------
문장: 행정구가 동대문구이고 행정동이 휘경2동인 학교의 전화번호와 팩스번호를 찾아줘
생성된 쿼리: SELECT TEL, FAX
FROM SEBC_COLLEGE_INFO_KOR
WHERE H_KOR_GU = '동대문구' AND H_KOR_DONG = '휘경2동'
정답 쿼리: SELECT TEL, FAX FROM SEBC_COLLEGE_INFO_KOR WHERE H_KOR_GU = '동대문구' AND H_KOR_DONG = '휘경2동'
평가 결과: